# Feature Engineering
Here we create new row-wise features. Note: We DO NOT use `pd.get_dummies` here to prevent one-hot encoding data leakage. Categorical encoding will happen in the Pipeline!

In [2]:
import pandas as pd
import numpy as np

# keep_default_na=False to preserve our 'None' strings
train_df = pd.read_csv('Ames_Housing/datasets/train_cleaned.csv', keep_default_na=False, na_values=[''])
test_df = pd.read_csv('Ames_Housing/datasets/test_cleaned.csv', keep_default_na=False, na_values=[''])

In [3]:
def feature_engineer(df, is_train=True):
    # 1. Total SF
    df['Total SF'] = df['Total Bsmt SF'] + df['1st Flr SF'] + df['2nd Flr SF']
    
    # 2. Total Bathrooms
    df['Total Bathrooms'] = (df['Full Bath'] + (0.5 * df['Half Bath']) + 
                             df['Bsmt Full Bath'] + (0.5 * df['Bsmt Half Bath']))
    
    # 3. House Age
    df['House Age'] = df['Yr Sold'].astype(int) - df['Year Built']
    df['Years Since Remodel'] = df['Yr Sold'].astype(int) - df['Year Remod/Add']
    
    # 4. Binary Features
    df['Has Pool'] = df['Pool Area'].apply(lambda x: 1 if x > 0 else 0)
    
    # 5. Drop original columns
    cols_to_drop = [
        'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 
        'Full Bath', 'Half Bath', 'Bsmt Full Bath', 'Bsmt Half Bath',
        'Year Built', 'Year Remod/Add'
    ]
    df = df.drop(cols_to_drop, axis=1)
    
    # 6. Ordinal Encoding
    qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    qual_cols = [
        'Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond', 
        'Heating QC', 'Kitchen Qual', 'Fireplace Qu', 
        'Garage Qual', 'Garage Cond', 'Pool QC'
    ]
    for col in qual_cols:
        if col in df.columns:
            df[col] = df[col].map(qual_map).fillna(0)
            
    exposure_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
    if 'Bsmt Exposure' in df.columns:
        df['Bsmt Exposure'] = df['Bsmt Exposure'].map(exposure_map).fillna(0)
        
    bsmt_fin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
    if 'BsmtFin Type 1' in df.columns:
        df['BsmtFin Type 1'] = df['BsmtFin Type 1'].map(bsmt_fin_map).fillna(0)
    if 'BsmtFin Type 2' in df.columns:
        df['BsmtFin Type 2'] = df['BsmtFin Type 2'].map(bsmt_fin_map).fillna(0)
        
    # 7. Log Transform Target (ONLY on the Training Set!)
    # We do not transform the test target here, we will do it at evaluation time if needed.
    if is_train and 'SalePrice' in df.columns:
        df['SalePrice'] = np.log1p(df['SalePrice'])
        
    return df

train_fe = feature_engineer(train_df, is_train=True)
test_fe = feature_engineer(test_df, is_train=False)


In [5]:
train_fe.to_csv('Ames_Housing/datasets/train_featured.csv', index=False)
test_fe.to_csv('Ames_Housing/datasets/test_featured.csv', index=False)
print("Feature Engineering Complete. Saved to datasets/")

Feature Engineering Complete. Saved to datasets/
